# Lekcja 4. Wybieranie i czyszczenie danych

**Czas:** 90–120 minut.

## Po tej lekcji potrafisz

- wybrać kolumny, filtrować i sortować wiersze,
- wykryć duplikaty i braki oraz uzasadnić ich obsługę,
- zamienić tekst ceny i daty na właściwe typy,
- zapisać oczyszczony CSV i policzyć odrzucone rekordy.

Pracuj od góry do dołu. Uruchom komórkę przez **Shift+Enter**. Najpierw przewiduj wynik, potem uruchamiaj kod. W ćwiczeniach zastąp komentarze własnym kodem. Rozwiązania otwieraj po samodzielnej próbie.


In [ ]:
from pathlib import Path

# Notebook działa po uruchomieniu z katalogu kursu lub z jego podkatalogu.
KATALOG_KURSU = next(
    katalog for katalog in [Path.cwd(), *Path.cwd().parents]
    if (katalog / "dane" / "sprzedaz_kawiarni.csv").exists()
)
DANE = KATALOG_KURSU / "dane"
WYNIKI = KATALOG_KURSU / "wyniki"
WYNIKI.mkdir(exist_ok=True)

import pandas as pd
surowe = pd.read_csv(DANE / "sprzedaz_kawiarni.csv")
sprzedaz = surowe.copy()


## 1. Wybierz i przefiltruj

Pojedyncze nawiasy wybierają kolumnę; lista nazw wewnątrz nawiasów wybiera tabelę. Porównanie całej kolumny tworzy maskę wartości `True` i `False`.


In [ ]:
display(sprzedaz[["produkt", "liczba_sztuk"]].head())
wieksze_zamowienia = sprzedaz[sprzedaz["liczba_sztuk"] >= 3]
display(wieksze_zamowienia.head())


### Ćwiczenie: Sprzedaż na wynos

Wybierz wiersze z kanałem „na wynos”. Wyświetl kolumny `produkt`, `kanal`, `liczba_sztuk`, sortując po ilości malejąco.

<details><summary>Podpowiedź</summary>

Użyj `== "na wynos"` i `.sort_values("liczba_sztuk", ascending=False)`.

</details>


In [ ]:
# Twoje rozwiązanie:
# Zastąp ten komentarz własnym kodem.


**Sprawdź swoją odpowiedź:** W każdym pokazanym wierszu kanał to „na wynos”, a pierwsze ilości wynoszą 4.


## 2. Duplikaty

W tym zbiorze każda pozycja ma unikalne ID. Powtórzone ID jest błędem eksportu. W innym zbiorze dwa podobne wiersze mogą opisywać dwie prawdziwe transakcje — najpierw ustal regułę.


In [ ]:
print("Przed:", len(sprzedaz))
print("Powtórzone ID:", sprzedaz.duplicated(subset="id_transakcji").sum())
sprzedaz = sprzedaz.drop_duplicates(subset="id_transakcji").copy()
print("Po:", len(sprzedaz))


## 3. Uporządkuj tekst i ceny

Metody `.str` pracują na tekście w całej kolumnie. `.astype(float)` konwertuje poprawny zapis liczby. W naszym pliku wszystkie niepuste ceny mają poprawny format po zamianie przecinka.


In [ ]:
sprzedaz["produkt"] = sprzedaz["produkt"].str.strip()
sprzedaz["cena_jednostkowa"] = (
    sprzedaz["cena_jednostkowa"].str.replace(",", ".", regex=False).astype(float)
)
display(sprzedaz[["produkt", "cena_jednostkowa"]].head())


## 4. Brak kategorii można odtworzyć

W tym ćwiczeniu znamy przypisanie produktów do kategorii ze słownika danych. Nie zgadujemy brakujących wartości.


In [ ]:
kategorie_produktow = {
    "Espresso": "napoje", "Latte": "napoje", "Herbata": "napoje",
    "Sernik": "jedzenie", "Croissant": "jedzenie", "Kanapka": "jedzenie",
}
print("Braki przed:", sprzedaz["kategoria"].isna().sum())
sprzedaz["kategoria"] = sprzedaz["kategoria"].fillna(
    sprzedaz["produkt"].map(kategorie_produktow)
)
print("Braki po:", sprzedaz["kategoria"].isna().sum())


## 5. Nieznana ilość i błędna data

Nieznana ilość nie oznacza zera. `errors="coerce"` zamienia niepoprawną datę na `NaT`, czyli brak daty. Zachowujemy odrzucone rekordy, żeby decyzja była widoczna.


In [ ]:
sprzedaz["data"] = pd.to_datetime(sprzedaz["data"], format="%Y-%m-%d", errors="coerce")
niepelne = sprzedaz["liczba_sztuk"].isna() | sprzedaz["data"].isna()
odrzucone = sprzedaz[niepelne].copy()
sprzedaz = sprzedaz[~niepelne].copy()
sprzedaz["liczba_sztuk"] = sprzedaz["liczba_sztuk"].astype(int)
print("Odrzucone po usunięciu duplikatów:", len(odrzucone))
print("Pozostałe:", len(sprzedaz))
display(odrzucone)


`|` oznacza „lub”, `&` oznacza „i”, a `~` odwraca maskę. Przy łączeniu porównań używaj nawiasów, np. `(sprzedaz["liczba_sztuk"] >= 2) & (sprzedaz["kanal"] == "na wynos")`. Usunięcie braków jest decyzją na potrzeby ćwiczenia. Wyniki opisują zachowane rekordy; nie znamy pełnej wartości utraconej sprzedaży.


### Ćwiczenie: Kontrola i filtr z dwoma warunkami

Sprawdź braki i duplikaty w oczyszczonych danych. Wybierz sprzedaż co najmniej dwóch sztuk na wynos.

<details><summary>Podpowiedź</summary>

Użyj `.isna().sum()`, `.duplicated(subset="id_transakcji").sum()` oraz dwóch porównań połączonych `&`.

</details>


In [ ]:
# Twoje rozwiązanie:
# Zastąp ten komentarz własnym kodem.


**Sprawdź swoją odpowiedź:** Po pełnym czyszczeniu zostaje **236 wierszy**, bez braków i powtórzonych ID. Wybrany podzbiór spełnia oba warunki.


## 6. Zapisz wynik

`index=False` zapobiega zapisaniu dodatkowej kolumny z numerami wierszy. Kolejne lekcje mają przygotowaną kopię czystych danych, więc można otwierać je niezależnie.


In [ ]:
sprzedaz.to_csv(WYNIKI / "moja_sprzedaz_czysta.csv", index=False)
odrzucone.to_csv(WYNIKI / "odrzucone.csv", index=False)
print("Zapisano pliki w:", WYNIKI)


## Zatrzymaj się i wyjaśnij

Dlaczego brakującej ilości nie uzupełniamy zerem? Co uzasadnia usuwanie powtórzonych ID? Jakie ryzyko niesie odrzucanie wierszy?

## Krótka praca własna (15–25 minut)

Zapisz raport czyszczenia: 243 rekordy wejściowe → 3 duplikaty → 4 rekordy bez ilości lub poprawnej daty → 236 rekordów. Podaj regułę dla każdego kroku. Porównaj własny zapis z `dane/sprzedaz_kawiarni_gotowa.csv`.

## Przed zamknięciem

Zapisz notebook. Wybierz **Kernel → Restart Kernel and Run All Cells** (restart jądra i uruchomienie wszystkich komórek). Jeżeli kod działa tylko po uruchomieniu komórek w szczególnej kolejności, uporządkuj go. Następnie zapisz ponownie.

Przykładowe odpowiedzi znajdziesz w odpowiadającym tej lekcji pliku w katalogu `rozwiazania/`.
